# Rede logística e risco de SLA no e-commerce

## Apresentação da fonte e pré-análise

Este notebook apresenta a fonte Olist (Olist, 2018) e verifica se ela sustenta a pergunta de pesquisa. A lógica de leitura e cálculo está em `src/ecom_delivery_sla`; o notebook permanece como uma interface narrativa e reproduzível.

Pergunta central : 
- *É possível identificar padrões de atraso e de violação de SLA a partir das características conhecidas no momento da compra, da relação entre vendedores e destinos e da estrutura temporal observada na rede?*

A unidade de análise será o pedido. O atraso contínuo e a violação binária serão construídos pelo pacote do projeto.

## Escopo mensurável e limites

Nesta etapa medimos diretamente o cumprimento do prazo de entrega no nível do pedido: `delivery_delay_days` compara a data real com a data estimada, e `sla_breach` identifica atraso positivo. Também podemos descrever prazo prometido, prazo realizado, sazonalidade, destino, categoria, composição de vendedores e distância aproximada entre vendedor e cliente.

Essas variáveis não transformam o dataset em um rastreamento logístico completo. A fonte não registra entrega individual por vendedor ou item, localização de hubs, rota efetiva, modal, motivo operacional do atraso ou eventos intermediários. Portanto, o projeto medirá risco histórico de descumprimento do prazo prometido; indicadores de vendedor e geolocalização serão proxies, e não medições diretas de um SLA individual.

## Fonte e inventário observável

Antes de discutir modelos, precisamos verificar quais tabelas estão disponíveis, qual é a ordem de grandeza da fonte, quais campos cada uma possui e quais tipos o pandas observa ao ler os CSVs. Esta leitura é um inventário estrutural; ela não substitui os contratos da tabela analítica.

In [1]:
from pathlib import Path
import sys

from IPython.display import display

LANG = "pt"
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ecom_delivery_sla import TABLE_FILES, schema_summary
from ecom_delivery_sla import build_supervised_orders, inventory_summary, read_olist_tables, target_summary

DATA_DIR = PROJECT_ROOT / 'data' / 'raw' / 'olist' / 'csv'
tables = read_olist_tables(DATA_DIR, table_names=TABLE_FILES)
display(inventory_summary(tables, language=LANG))
display(schema_summary(tables))

,tabela,linhas,colunas
0,pedidos,99441,8
1,clientes,99441,5
2,itens,112650,7
3,produtos,32951,9
4,pagamentos,103886,5
5,avaliações,99224,7
6,vendedores,3095,4
7,geolocalização,1000163,5
8,tradução de categorias,71,2


,dataset,rows,field_order,field,data_type,non_null_pct
0,orders,99441,1,order_id,object,1.000000
1,orders,99441,2,customer_id,object,1.000000
2,orders,99441,3,order_status,object,1.000000
3,orders,99441,4,order_purchase_timestamp,object,1.000000
4,orders,99441,5,order_approved_at,object,0.998391
5,orders,99441,6,order_delivered_carrier_date,object,0.982070
6,orders,99441,7,order_delivered_customer_date,object,0.970183
7,orders,99441,8,order_estimated_delivery_date,object,1.000000
8,customers,99441,1,customer_id,object,1.000000
9,customers,99441,2,customer_unique_id,object,1.000000


### Insight do inventário

A fonte disponibiliza tabelas de pedidos, itens, clientes, vendedores, produtos, pagamentos, avaliações, geolocalização e tradução. A tabela estrutural explicita os campos, o tipo observado pelo pandas e o preenchimento de cada campo. Os tipos são tipos de leitura do CSV — por exemplo, timestamps ainda podem aparecer como `object` antes da conversão — e não substituem a definição semântica dos dados. Essa composição é suficiente para investigar o processo de entrega em nível de pedido e enriquecer a análise com relações entre origem e destino, mas ainda não prova que toda variável estava disponível no momento da compra.

## População supervisionada e definição inicial do alvo

Agora filtramos os pedidos entregues com datas de entrega real e estimada observáveis. Em seguida, calculamos o atraso em dias e a indicação binária de violação: `sla_breach = atraso > 0`. O objetivo é tornar explícito quantas observações realmente podem participar de uma análise supervisionada.

In [2]:
supervised = build_supervised_orders(tables['orders'])
display(target_summary(supervised, language=LANG))

,métrica,valor
0,Pedidos supervisionáveis,96470.000000
1,Violações de SLA,6534.000000
2,Taxa de violação,0.067731
3,Mediana do atraso (dias),-11.948102
4,P95 do atraso (dias),3.816993
5,P99 do atraso (dias),18.939553
6,Atraso > 30 dias,360.000000
7,Antecipação < -30 dias,2352.000000


### Insight do alvo

A população supervisionada contém 96.470 pedidos, e 6.534 violam pela regra de dia de calendário: uma taxa global de aproximadamente 6,77%. O alvo é, portanto, desbalanceado e a definição adotada precisa ser preservada em todos os experimentos. A taxa descreve o histórico observado; não representa automaticamente a probabilidade futura de atraso.

## Limites atuais

A pré-análise não autoriza conclusões causais nem modelagem final. Ainda precisamos definir a tabela analítica, o ponto de corte temporal, a política para pedidos com múltiplos vendedores e a construção de métricas históricas da rede.

Até a conclusão dos contratos, `production_eligible = false`.

Fonte: OLIST. *Brazilian E-Commerce Public Dataset by Olist*. Kaggle.

## Referências

Olist (2018). *Brazilian E-Commerce Public Dataset by Olist*. Kaggle. https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce